# Viper GPU worker v3 (leave this tab open)

Runtime > Change runtime type > **GPU** (+ High RAM), then run the cell below once and leave the tab open.

Polls `MyDrive/viper_ml/jobs/` for `*.json` jobs written by the laptop and writes `MyDrive/viper_ml/results/<job>/`.
Two job kinds:
- training: `{"code": "code.zip", "data": "data_v1.zip", "args": "--epochs 12"}` runs `ml/train.py`
- any script: `{"code": "code_rl.zip", "script": "ml/rl_train.py", "args": "...", "pip": "unswbc"}`
  (`{OUT}` in args is replaced by the job's results folder; the zip is unpacked as the working directory).

v3: the laptop can stop the running job by creating `jobs/STOP_<job name>` (or `jobs/STOP_ALL`).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import glob, json, os, shutil, subprocess, time, datetime, zipfile, signal
ROOT = '/content/drive/MyDrive/viper_ml'
for d in ('jobs', 'results'): os.makedirs(f'{ROOT}/{d}', exist_ok=True)
if shutil.which('nvidia-smi'):   # absent on CPU runtimes
    print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True).stdout)
print('cpus', os.cpu_count())

def unpack(zip_name, dest):
    marker = f'{dest}/.from_{zip_name}_{os.path.getmtime(f"{ROOT}/{zip_name}")}'
    if os.path.exists(marker): return
    shutil.rmtree(dest, ignore_errors=True); os.makedirs(dest)
    zipfile.ZipFile(f'{ROOT}/{zip_name}').extractall(dest)
    open(marker, 'w').close()

def stop_requested(name):
    for f in (f'{ROOT}/jobs/STOP_{name}', f'{ROOT}/jobs/STOP_ALL'):
        if os.path.exists(f):
            os.remove(f); return True
    return False

installed = set()
while True:
    jobs = sorted(glob.glob(f'{ROOT}/jobs/*.json'))
    if not jobs:
        time.sleep(60); continue
    job = jobs[0]; name = os.path.basename(job)[:-5]
    running = job[:-5] + '.running'
    os.rename(job, running)
    spec = json.load(open(running))
    out = f'{ROOT}/results/{name}'; os.makedirs(out, exist_ok=True)
    print(datetime.datetime.now().strftime('%H:%M'), 'running', name, spec, flush=True)
    try:
        for pkg in spec.get('pip', '').split():
            if pkg not in installed:
                subprocess.run(f'pip install -q {pkg}', shell=True); installed.add(pkg)
        code_dir = '/content/code_' + spec['code'][:-4]
        unpack(spec['code'], code_dir)
        if 'script' in spec:
            args = spec.get('args', '').replace('{OUT}', out)
            cmd = f"cd {code_dir} && python -u {spec['script']} {args}"
        else:
            unpack(spec['data'], '/content/data/' + spec['data'][:-4])
            cmd = f"cd {code_dir} && python ml/train.py --data /content/data/{spec['data'][:-4]} --out {out}/model.pt {spec.get('args', '')}"
        with open(f'{out}/worker_log.txt', 'w') as log:
            p = subprocess.Popen(cmd, shell=True, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
            while p.poll() is None:
                time.sleep(20)
                if stop_requested(name):
                    os.killpg(p.pid, signal.SIGKILL); p.wait()
                    print(datetime.datetime.now().strftime('%H:%M'), 'stopped', name, flush=True)
            rc = p.returncode
        open(f'{out}/done', 'w').write(str(rc))
    except Exception as e:
        open(f'{out}/done', 'w').write('error ' + repr(e))
    os.remove(running)
    print(datetime.datetime.now().strftime('%H:%M'), 'finished', name, open(f'{out}/done').read(), flush=True)
